In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [3]:
voxmat = np.genfromtxt("./msc01_L_dts.csv", delimiter=",", dtype=float)

In [4]:
voxmat.shape

(29697, 363)

In [24]:
idlst = [*range(0, voxmat.shape[1], 10)]

In [25]:
len(idlst)

37

In [23]:
idlist[1,:]

array([  0,  10,  20,  30,  40,  50,  60,  70,  80,  90, 100, 110, 120,
       130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250,
       260, 270, 280, 290, 300, 310, 320, 330, 340, 350, 360])

In [35]:
corr_t0 = []
for i in range(1,voxmat.shape[0]-2):
    for j in range(len(idlst)-2):
        corr_t0.append(stats.pearsonr(voxmat[i,idlst[j]:idlst[j+1]], voxmat[i+1,idlst[j]:idlst[j+1]]))

KeyboardInterrupt: 

In [40]:
vxt_list = []
for i in range(1,voxmat.shape[0]):
    for j in range(len(idlst)-2):
        vxt_list.append(voxmat[i,idlst[j]:idlst[j+1]])


In [41]:
len(vxt_list)/(voxmat.shape[0]-1)

35.0

In [46]:
# corrmat = np.zeros((len(vxt_list),len(vxt_list)))
corr_v1_t0 = []
for i, arr in enumerate(vxt_list):
    r,p = stats.pearsonr(vxt_list[0], arr)
    if (p<0.001) & (np.abs(r)>.7):
        corr_v1_t0.append((i,r,p))

In [48]:
len(corr_v1_t0)

39243

In [45]:
for k in range(1,voxmat.shape[0]-2,1000):
    namelist = {}
    for i in range(1000):
        for j in range(len(idlst)-2):
            namelist["voxel_"+str(i+k)+"_timestep_"+str(j)] = np.zeros(len(vxt_list[0]))
    df = pd.DataFrame.from_dict(namelist)

3

In [49]:
print(i,j,k)

NameError: name 'k' is not defined

In [50]:
for i in range(10):
    for j in range(i,10):
        print(i,j)

0 0
0 1
0 2
0 3
0 4
0 5
0 6
0 7
0 8
0 9
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
2 2
2 3
2 4
2 5
2 6
2 7
2 8
2 9
3 3
3 4
3 5
3 6
3 7
3 8
3 9
4 4
4 5
4 6
4 7
4 8
4 9
5 5
5 6
5 7
5 8
5 9
6 6
6 7
6 8
6 9
7 7
7 8
7 9
8 8
8 9
9 9


In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import corr_helper
import multiprocessing as mp 
from functools import partial

global subj, func, vts, vtn, count

In [3]:
def init_pool(p):
    global pool
    pool = p

In [4]:
def inner(v0):
    global pool, subj, func, vts, count
    
    print("Creating empty dataframe...", flush=True)

    vtdf = corr_helper.gen_corrdf(vts)
    
    print("Running inner loop...", flush=True)

    initializedcorr = partial(stats.pearsonr, x=v0)
    res = pool.map(initializedcorr, vts)

    vtdf.iloc[:,0] = res[:,0]
    vtdf.iloc[:,1] = res[:,1]
    count += 1
    print("Saving file...\n")
    vtdf.to_csv("./correlations/msc0"+str(subj)+"_sess0"+str(func)+"_L_corrdf_"+str(count)+".csv")

In [5]:
def outer(vlst):
    global pool
    print("Running outer loop...", flush=True)
    pool.map_async(inner, (vlst,))
    print("Called inner loop...", flush=True)

In [15]:
# if __name__ == '__main__':
    
# global subj, func, vts, vtn, count, pool

print("Initializing pool...", flush=True)

# create a pool object with initializer
p = mp.Pool(mp.cpu_count() - 1)
init_pool(p)

subj=6
func=1

print("Now running subject %s, session %s" % (str(subj), str(func)), flush=True)

cname = "./msc0"+str(subj)+"_sess0"+str(func)+"_L_dts.csv"
voxmat, idxlist = corr_helper.get_voxt_mat(cname)
count = 0

vts, vtn = corr_helper.gen_voxt_seg(voxmat,idxlist)
vts = vts[:2] # Keeps things simple; delete after debugging

p.apply_async(inner, vts)

p.close()

Initializing pool...
Now running subject 6, session 1


In [7]:
dir()

['In',
 'Out',
 '_',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'cname',
 'corr_helper',
 'count',
 'exit',
 'func',
 'get_ipython',
 'idxlist',
 'init_pool',
 'inner',
 'mp',
 'np',
 'open',
 'outer',
 'p',
 'partial',
 'pd',
 'pool',
 'quit',
 'stats',
 'subj',
 'voxmat',
 'vtn',
 'vts']

In [14]:
len(vts[0])

10